# Pipeline de Procesamiento de Datos con PySpark

Este notebook implementa un pipeline completo de procesamiento de datos utilizando PySpark.

**Objetivo:** Procesar datos de ventas y clientes para obtener insights de negocio.

**Etapas del pipeline:**
1. Importación de librerías
2. Generación de datos sintéticos
3. Exploración inicial
4. Limpieza de datos
5. Transformaciones
6. Agregaciones
7. Joins entre datasets
8. Análisis avanzado
9. Persistencia de resultados
10. Visualización final

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum, count, when, upper, trim, to_date, datediff, current_date, lit
from pyspark.sql.functions import round as spark_round
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
import random
from datetime import datetime, timedelta

print("✓ Librerías importadas correctamente")

In [0]:
# Generar datos de ventas (800 registros)
ventas_data = []
productos = ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Webcam', 'Auriculares', 'SSD', 'RAM']
regiones = ['Norte', 'Sur', 'Este', 'Oeste', 'Centro']

for i in range(800):
    fecha = datetime(2026, 1, 1) + timedelta(days=random.randint(0, 130))
    precio = float(f"{random.uniform(50, 2000):.2f}")
    ventas_data.append((
        i + 1,
        random.randint(1, 200),
        random.choice(productos),
        random.randint(1, 5),
        precio,
        fecha.strftime('%Y-%m-%d'),
        random.choice(regiones)
    ))

df_ventas = spark.createDataFrame(ventas_data, 
    ['venta_id', 'cliente_id', 'producto', 'cantidad', 'precio_unitario', 'fecha', 'region'])

# Generar datos de clientes (200 registros)
clientes_data = []
nombres = ['Juan', 'María', 'Carlos', 'Ana', 'Luis', 'Carmen', 'Pedro', 'Laura']
apellidos = ['García', 'Rodríguez', 'López', 'Martínez', 'González', 'Pérez', 'Sánchez', 'Ramírez']
categorias = ['Premium', 'Regular', 'Nuevo', 'VIP']

for i in range(200):
    clientes_data.append((
        i + 1,
        f"{random.choice(nombres)} {random.choice(apellidos)}",
        random.choice(categorias),
        random.choice([True, False]),
        random.randint(0, 50)
    ))

df_clientes = spark.createDataFrame(clientes_data,
    ['cliente_id', 'nombre', 'categoria', 'activo', 'compras_historicas'])

print(f"✓ Datos generados: {df_ventas.count()} ventas y {df_clientes.count()} clientes")

In [0]:
print("=== ESTRUCTURA DE VENTAS ===")
df_ventas.printSchema()
print(f"\nTotal de registros: {df_ventas.count()}")
print("\nPrimeras 5 filas:")
display(df_ventas.limit(5))

print("\n=== ESTRUCTURA DE CLIENTES ===")
df_clientes.printSchema()
print(f"\nTotal de registros: {df_clientes.count()}")
print("\nPrimeras 5 filas:")
display(df_clientes.limit(5))

In [0]:
# Limpiar datos de ventas
df_ventas_limpio = df_ventas \
    .filter(col('cantidad') > 0) \
    .filter(col('precio_unitario') > 0) \
    .withColumn('region', upper(trim(col('region')))) \
    .withColumn('fecha', to_date(col('fecha'), 'yyyy-MM-dd')) \
    .dropDuplicates(['venta_id'])

# Limpiar datos de clientes
df_clientes_limpio = df_clientes \
    .filter(col('nombre').isNotNull()) \
    .withColumn('nombre', trim(col('nombre'))) \
    .withColumn('categoria', upper(trim(col('categoria')))) \
    .dropDuplicates(['cliente_id'])

print(f"✓ Ventas después de limpieza: {df_ventas_limpio.count()} registros")
print(f"✓ Clientes después de limpieza: {df_clientes_limpio.count()} registros")

display(df_ventas_limpio.limit(5))

In [0]:
# Calcular total de venta y crear nuevas columnas
df_ventas_transformado = df_ventas_limpio \
    .withColumn('total_venta', spark_round(col('cantidad') * col('precio_unitario'), 2)) \
    .withColumn('dias_desde_venta', datediff(current_date(), col('fecha'))) \
    .withColumn('tipo_venta', 
        when(col('total_venta') > 5000, 'Alta')
        .when(col('total_venta') > 1000, 'Media')
        .otherwise('Baja')
    )

print("✓ Transformaciones aplicadas")
print("\nVentas con nuevas columnas calculadas:")
display(df_ventas_transformado.select('venta_id', 'producto', 'cantidad', 'precio_unitario', 'total_venta', 'tipo_venta').limit(10))

In [0]:
# Agregaciones por producto
df_por_producto = df_ventas_transformado \
    .groupBy('producto') \
    .agg(
        count('venta_id').alias('num_ventas'),
        sum('cantidad').alias('unidades_vendidas'),
        spark_round(sum('total_venta'), 2).alias('ingresos_totales'),
        spark_round(avg('precio_unitario'), 2).alias('precio_promedio')
    ) \
    .orderBy(col('ingresos_totales').desc())

print("=== VENTAS POR PRODUCTO ===")
display(df_por_producto)

# Agregaciones por región
df_por_region = df_ventas_transformado \
    .groupBy('region') \
    .agg(
        count('venta_id').alias('num_ventas'),
        spark_round(sum('total_venta'), 2).alias('ingresos_totales'),
        spark_round(avg('total_venta'), 2).alias('ticket_promedio')
    ) \
    .orderBy(col('ingresos_totales').desc())

print("\n=== VENTAS POR REGIÓN ===")
display(df_por_region)

In [0]:
# Join entre ventas y clientes
df_completo = df_ventas_transformado \
    .join(df_clientes_limpio, 'cliente_id', 'left') \
    .select(
        'venta_id',
        'cliente_id',
        'nombre',
        'categoria',
        'activo',
        'producto',
        'cantidad',
        'total_venta',
        'tipo_venta',
        'region',
        'fecha'
    )

print(f"✓ Join completado: {df_completo.count()} registros")
print("\nDatos combinados de ventas y clientes:")
display(df_completo.limit(10))

In [0]:
# Análisis del perfil de clientes
df_perfil_clientes = df_completo \
    .groupBy('cliente_id', 'nombre', 'categoria', 'activo') \
    .agg(
        count('venta_id').alias('total_compras'),
        spark_round(sum('total_venta'), 2).alias('valor_total_compras'),
        spark_round(avg('total_venta'), 2).alias('ticket_promedio'),
        count(when(col('tipo_venta') == 'Alta', 1)).alias('compras_alta_valor')
    ) \
    .withColumn('cliente_valioso', 
        when((col('valor_total_compras') > 10000) | (col('compras_alta_valor') > 2), 'Sí')
        .otherwise('No')
    ) \
    .orderBy(col('valor_total_compras').desc())

print("=== TOP 10 CLIENTES POR VALOR ===")
display(df_perfil_clientes.limit(10))

# Estadísticas por categoría de cliente
df_stats_categoria = df_perfil_clientes \
    .groupBy('categoria') \
    .agg(
        count('cliente_id').alias('num_clientes'),
        spark_round(avg('valor_total_compras'), 2).alias('valor_promedio'),
        spark_round(sum('valor_total_compras'), 2).alias('valor_total')
    ) \
    .orderBy(col('valor_total').desc())

print("\n=== ANÁLISIS POR CATEGORÍA DE CLIENTE ===")
display(df_stats_categoria)

In [0]:
# Calcular métricas clave del negocio
total_ventas = df_ventas_transformado.count()
ingresos_totales = df_ventas_transformado.agg(sum('total_venta')).collect()[0][0]
ticket_promedio = df_ventas_transformado.agg(avg('total_venta')).collect()[0][0]
clientes_activos = df_clientes_limpio.filter(col('activo') == True).count()

# Crear DataFrame de resumen (todos los valores como float para consistencia)
resumen_data = [
    ('Total de Ventas', float(total_ventas)),
    ('Ingresos Totales', float(f"{ingresos_totales:.2f}")),
    ('Ticket Promedio', float(f"{ticket_promedio:.2f}")),
    ('Clientes Activos', float(clientes_activos)),
    ('Clientes Totales', float(df_clientes_limpio.count()))
]

df_resumen = spark.createDataFrame(resumen_data, ['Métrica', 'Valor'])

print("="*50)
print("RESUMEN DEL PIPELINE DE PROCESAMIENTO")
print("="*50)
display(df_resumen)

print(f"\n✓ Pipeline completado exitosamente")
print(f"✓ Total de registros procesados: {total_ventas} ventas")
print(f"✓ Ingresos totales generados: ${ingresos_totales:,.2f}")